# VisML Lab: Explaining a Text Classifier with LIME
DS-GA 3001, Fall 2026. Work in pairs. For each step: **predict first, then run.**

We train a simple classifier on two newsgroups (`alt.atheism` vs `soc.religion.christian`), then ask LIME *which words* pushed one prediction toward one class. The question for today: **does the model get the right answer for the right reasons?**

Run the cells top to bottom (Runtime > Run all is fine the first time).

In [ ]:
%pip install -q lime

## 1. Data and model

In [ ]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from lime.lime_text import LimeTextExplainer
from IPython.display import display, HTML

cats = ['alt.atheism', 'soc.religion.christian']
class_names = ['atheism', 'christian']

# Version A: the raw posts, headers and signatures included
train = fetch_20newsgroups(subset='train', categories=cats)
test = fetch_20newsgroups(subset='test', categories=cats)

pipe = make_pipeline(TfidfVectorizer(lowercase=True), LogisticRegression(max_iter=1000))
pipe.fit(train.data, train.target)
print('Test accuracy:', round(accuracy_score(test.target, pipe.predict(test.data)), 3))

## 2. Explain one prediction
LIME removes random words from the text, asks the model again, and fits a small linear model to those answers. The weights of that small model are the explanation. We explain the **christian** class: **positive weight = pushes toward christian, negative weight = pushes toward atheism.**

In [ ]:
explainer = LimeTextExplainer(class_names=class_names)

def explain(model, text, k=8):
    return explainer.explain_instance(text, model.predict_proba, num_features=k)

def show(exp):
    display(HTML(exp.as_html(text=True)))   # works in Colab and in recent IPython

idx = 83                      # change this number to look at other posts
text, truth = test.data[idx], class_names[test.target[idx]]
probs = pipe.predict_proba([text])[0]
print(f'True class: {truth}   Model says: {{c: round(float(p), 3) for c, p in zip(class_names, probs)}}')

show(explain(pipe, text))

**Predict, then look at the highlighted words.**

* Which words would *you* have used to decide between atheism and christian?
* Are the words LIME highlights the ones you expected? Write down any that surprise you.

In [ ]:
# Top words for several posts at once. Positive weight = toward christian, negative = toward atheism.
for i in [5, 20, 83, 140, 200]:
    t = test.data[i]
    e = explain(pipe, t)
    print(f'--- post {i}: true={class_names[test.target[i]]}, '
          f'predicted={class_names[pipe.predict([t])[0]]}')
    print([(w, round(float(s), 3)) for w, s in e.as_list()])

## 3. Exercise 1: find a right answer for the wrong reason
Look at the weights printed above and in the highlighted text. Find a post where the model is **correct** but a highlighted word has **nothing to do with religion** (a header field, an email domain, a name).

Record: the post number, the word, its weight.

## 4. Exercise 2: take the shortcut away
Reload the data with headers, footers and quoted replies removed. The text that remains is only what the author wrote.

In [ ]:
train_c = fetch_20newsgroups(subset='train', categories=cats, remove=('headers', 'footers', 'quotes'))
test_c = fetch_20newsgroups(subset='test', categories=cats, remove=('headers', 'footers', 'quotes'))

pipe_c = make_pipeline(TfidfVectorizer(lowercase=True), LogisticRegression(max_iter=1000))
pipe_c.fit(train_c.data, train_c.target)
print('Test accuracy (cleaned):', round(accuracy_score(test_c.target, pipe_c.predict(test_c.data)), 3))

**Predict:** will accuracy go up, down, or stay the same? Compare with the number from Section 1. Then explain the same post again.

In [ ]:
# Same post index as before, now with the cleaned model
text_c, truth_c = test_c.data[idx], class_names[test_c.target[idx]]
print('True class:', truth_c, '  Model says:', {c: round(float(p), 3) for c, p in zip(class_names, pipe_c.predict_proba([text_c])[0])})
show(explain(pipe_c, text_c))

* Did the accuracy change? Did the *explanation* change?
* Which model would you rather deploy on new text? Which one looks better on the leaderboard?

## 5. Exercise 3: probe it yourself
Write your own short text and see how the model reads it. Try a sentence that mixes both topics, or one that mentions neither.

In [ ]:
my_text = "Replace this with your own sentence about faith, evidence, or anything else."

print({c: round(float(p), 3) for c, p in zip(class_names, pipe_c.predict_proba([my_text])[0])})
show(explain(pipe_c, my_text))

## 6. Compare with the model's own weights
For a linear model we can read the weights directly. How well does LIME's local view agree with the global one?

In [ ]:
vec, clf = pipe_c.steps[0][1], pipe_c.steps[1][1]
words = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])
print('Most atheism-leaning words:', [str(w) for w in words[order[:15]]])
print('Most christian-leaning words:', [str(w) for w in words[order[-15:]]])

## Discuss
1. Where did LIME's answer match your intuition, and where did it not?
2. LIME explains one prediction with a *simple* local model. When could that be misleading? (Hint: rerun the explanation of the same post a few times and compare the weights.)
3. How would you show these explanations to a non-technical user? What would you change in the visualization?

Reference: Ribeiro, Singh, Guestrin (2016), *"Why Should I Trust You?": Explaining the Predictions of Any Classifier*, KDD. [arXiv:1602.04938](https://arxiv.org/abs/1602.04938)